# Step 5 V5: Feature Representation Ablation

This notebook follows the V5 experiment design recorded in `AGENT.md`.

The current pipeline is:

```text
UofTDB 5-second segments
-> train a UofTDB-specific ResNet1D for 1019-class subject classification
-> remove the final classification layer
-> keep the embedding before the classifier
-> generate fiducial, embedding, and combined feature sets
-> use these features in later RP and RanPAC steps
```

This notebook does not run Random Projection or RanPAC. It does not use random CNN embeddings or the external `ecgid_resnet1d.pth` as the main embedding backbone.


## 0. Paths, Dependencies, and Global Configuration

This section sets paths, random seeds, and basic dependencies. If Codex bundled Python is used, extra dependencies are added from the project `.python_packages` directory into `sys.path`. This only affects the runtime import path and does not change Step 3 or Step 4 output files.


In [3]:
from pathlib import Path
import json
import math
import os
import random
import sys
import time
import warnings

ROOT = Path(r"F:\ECG")
EXTRA_SITE_PACKAGES = ROOT / ".python_packages"
if EXTRA_SITE_PACKAGES.exists() and str(EXTRA_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(EXTRA_SITE_PACKAGES))

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import neurokit2 as nk
    NK_AVAILABLE = True
    NEUROKIT_VERSION = nk.__version__
except Exception as error:
    nk = None
    NK_AVAILABLE = False
    NEUROKIT_VERSION = None
    print("NeuroKit2 failed; using scipy fallback. Error:", repr(error))

try:
    from scipy.signal import find_peaks
    SCIPY_AVAILABLE = True
except Exception as error:
    find_peaks = None
    SCIPY_AVAILABLE = False
    print("scipy fallback R-peak detection failed. Error:", repr(error))

from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "feature_outputs_v5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_SEGMENTS_PATH = PROCESSED_DIR / "X_segments_sit_5s.npy"
Y_SEGMENTS_PATH = PROCESSED_DIR / "y_segments_sit_5s.npy"
SEGMENT_METADATA_PATH = PROCESSED_DIR / "segment_metadata_sit_5s.csv"
STREAM_ORDER_PATH = PROCESSED_DIR / "stream_order_time_sit_5s.npy"
STREAM_METADATA_PATH = PROCESSED_DIR / "stream_metadata_time_sit_5s.csv"

LABEL_MAPPING_PATH = OUTPUT_DIR / "resnet1d_label_mapping.csv"
TRAIN_INDICES_PATH = OUTPUT_DIR / "resnet1d_train_indices.npy"
VAL_INDICES_PATH = OUTPUT_DIR / "resnet1d_val_indices.npy"
HOLDOUT_INDICES_PATH = OUTPUT_DIR / "resnet1d_holdout_indices.npy"
SPLIT_SUMMARY_PATH = OUTPUT_DIR / "resnet1d_split_summary.csv"

BEST_MODEL_PATH = OUTPUT_DIR / "resnet1d_uoftdb_best.pth"
TRAINING_LOG_PATH = OUTPUT_DIR / "resnet1d_training_log.csv"
CONFIG_PATH = OUTPUT_DIR / "resnet1d_config.json"
HOLDOUT_REPORT_PATH = OUTPUT_DIR / "resnet1d_holdout_report.csv"

EMBEDDING_FEATURE_PATH = OUTPUT_DIR / "X_features_embedding_resnet1d.npy"
EMBEDDING_REPORT_PATH = OUTPUT_DIR / "embedding_resnet1d_report.csv"

FIDUCIAL_FEATURE_PATH = OUTPUT_DIR / "X_features_fiducial_pqrst.npy"
FIDUCIAL_NAMES_PATH = OUTPUT_DIR / "fiducial_pqrst_feature_names.csv"
FIDUCIAL_QUALITY_PATH = OUTPUT_DIR / "fiducial_pqrst_quality_report.csv"

BOTH_FEATURE_PATH = OUTPUT_DIR / "X_features_both_pqrst_resnet1d.npy"
Y_STREAM_OUTPUT_PATH = OUTPUT_DIR / "y_stream_feature_ablation_v5.npy"
FEATURE_SUMMARY_PATH = OUTPUT_DIR / "feature_ablation_v5_summary.csv"

SEED = 27
SAMPLING_RATE = 200
EXPECTED_NUM_SAMPLES = 36103
EXPECTED_SIGNAL_LENGTH = 1000
EXPECTED_NUM_CLASSES = 1019

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    gpu_name = torch.cuda.get_device_name(0)
else:
    gpu_name = "cpu"

print("project root:", ROOT)
print("output directory:", OUTPUT_DIR)
print("PyTorch:", torch.__version__)
print("device:", device)
print("GPU:", gpu_name)
print("NeuroKit2 available:", NK_AVAILABLE, NEUROKIT_VERSION)
print("scipy fallback available:", SCIPY_AVAILABLE)


project root: F:\ECG
output directory: F:\ECG\data\processed\feature_outputs_v5
PyTorch: 2.11.0+cu128
device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
NeuroKit2 available: True 0.2.13
scipy fallback available: True


## 1. Reload Step 3 / Step 4 Outputs and Build the Time-aware Stream

This section reloads the Step 3 5-second segment files and the Step 4 time-aware stream files from disk. The main experiment stream is fixed as:

```python
X_stream = X_segments[stream_order_time]
y_stream = y_segments[stream_order_time]
```

The fiducial, embedding, and combined feature representations are all extracted from the same `X_stream` and `y_stream`.


In [6]:
input_paths = [
    X_SEGMENTS_PATH,
    Y_SEGMENTS_PATH,
    SEGMENT_METADATA_PATH,
    STREAM_ORDER_PATH,
    STREAM_METADATA_PATH,
]

for path in input_paths:
    print(path, "exists =", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

X_segments = np.load(X_SEGMENTS_PATH, mmap_mode="r")
y_segments = np.load(Y_SEGMENTS_PATH)
segment_metadata = pd.read_csv(SEGMENT_METADATA_PATH)
stream_order_time = np.load(STREAM_ORDER_PATH)
stream_metadata = pd.read_csv(STREAM_METADATA_PATH)

X_stream = X_segments[stream_order_time].astype(np.float32)
y_stream = y_segments[stream_order_time]

unique_subjects = np.sort(np.unique(y_stream))

x_nan_count = int(np.isnan(X_stream).sum())
x_inf_count = int(np.isinf(X_stream).sum())
y_nan_count = int(pd.isna(pd.Series(y_stream)).sum())

validation_rows = [
    {"check": "X_stream shape", "observed": str(X_stream.shape), "expected": "(36103, 1000)", "passed": X_stream.shape == (EXPECTED_NUM_SAMPLES, EXPECTED_SIGNAL_LENGTH)},
    {"check": "y_stream shape", "observed": str(y_stream.shape), "expected": "(36103,)", "passed": y_stream.shape == (EXPECTED_NUM_SAMPLES,)},
    {"check": "unique subjects", "observed": int(len(unique_subjects)), "expected": EXPECTED_NUM_CLASSES, "passed": len(unique_subjects) == EXPECTED_NUM_CLASSES},
    {"check": "X_stream NaN count", "observed": x_nan_count, "expected": 0, "passed": x_nan_count == 0},
    {"check": "X_stream Inf count", "observed": x_inf_count, "expected": 0, "passed": x_inf_count == 0},
    {"check": "y_stream NaN count", "observed": y_nan_count, "expected": 0, "passed": y_nan_count == 0},
    {"check": "segment metadata rows", "observed": len(segment_metadata), "expected": EXPECTED_NUM_SAMPLES, "passed": len(segment_metadata) == EXPECTED_NUM_SAMPLES},
    {"check": "stream metadata rows", "observed": len(stream_metadata), "expected": EXPECTED_NUM_SAMPLES, "passed": len(stream_metadata) == EXPECTED_NUM_SAMPLES},
    {"check": "stream order length", "observed": len(stream_order_time), "expected": EXPECTED_NUM_SAMPLES, "passed": len(stream_order_time) == EXPECTED_NUM_SAMPLES},
]

validation_df = pd.DataFrame(validation_rows)
print(validation_df.to_string(index=False))

if not bool(validation_df["passed"].all()):
    raise ValueError("Step 5 inputs are incomplete. Please check Step 3 / Step 4 outputs.")


F:\ECG\data\processed\X_segments_sit_5s.npy exists = True
F:\ECG\data\processed\y_segments_sit_5s.npy exists = True
F:\ECG\data\processed\segment_metadata_sit_5s.csv exists = True
F:\ECG\data\processed\stream_order_time_sit_5s.npy exists = True
F:\ECG\data\processed\stream_metadata_time_sit_5s.csv exists = True
                check      observed      expected  passed
       X_stream shape (36103, 1000) (36103, 1000)    True
       y_stream shape      (36103,)      (36103,)    True
      unique subjects          1019          1019    True
   X_stream NaN count             0             0    True
   X_stream Inf count             0             0    True
   y_stream NaN count             0             0    True
segment metadata rows         36103         36103    True
 stream metadata rows         36103         36103    True
  stream order length         36103         36103    True


## 2. Encode Subject Labels for ResNet1D Training and Later RanPAC

The ResNet1D training task is 1019-class subject classification. The original `subject_id` values are encoded as contiguous `class_index` values, and the mapping table is saved. The saved `y_stream_feature_ablation_v5.npy` uses this encoded class index so Step 6 and Step 7 can reuse the same labels.


In [9]:
label_mapping = pd.DataFrame({
    "subject_id": unique_subjects,
    "class_index": np.arange(len(unique_subjects), dtype=np.int64),
})
label_mapping.to_csv(LABEL_MAPPING_PATH, index=False)

subject_to_class = dict(zip(label_mapping["subject_id"].tolist(), label_mapping["class_index"].tolist()))
y_encoded = pd.Series(y_stream).map(subject_to_class).to_numpy(dtype=np.int64)

print("label mapping saved:", LABEL_MAPPING_PATH)
print("encoded labels shape:", y_encoded.shape)
print("encoded class range:", int(y_encoded.min()), "to", int(y_encoded.max()))
print("unique encoded classes:", len(np.unique(y_encoded)))

if len(np.unique(y_encoded)) != EXPECTED_NUM_CLASSES:
    raise ValueError("Label encoding produced an unexpected class count; expected 1019.")


label mapping saved: F:\ECG\data\processed\feature_outputs_v5\resnet1d_label_mapping.csv
encoded labels shape: (36103,)
encoded class range: 0 to 1018
unique encoded classes: 1019


## 3. Create a Stratified Split for Offline ResNet1D Backbone Training

This split is used only to train the offline ResNet1D embedding extractor:

```text
train = 70%
validation = 15%
holdout = 15%
seed = 27
stratify by subject label
```

This split is not the RanPAC online evaluation protocol. RanPAC later uses the full time-aware stream order and is evaluated with prequential test-then-train.

If some subjects have too few segments, strict sklearn stratification may fail. In that case, the notebook uses a rare-class-aware per-class fallback: classes that can be split into three groups still use 70/15/15, while singleton classes are placed in train only.


In [12]:
all_indices = np.arange(len(y_encoded), dtype=np.int64)
split_method = "sklearn_stratified_train_test_split"
rare_class_notes = []

def rare_class_stratified_fallback(labels, seed):
    rng = np.random.default_rng(seed)
    train_parts = []
    val_parts = []
    holdout_parts = []
    notes = []

    for class_id in np.sort(np.unique(labels)):
        class_indices = np.where(labels == class_id)[0]
        class_indices = class_indices.copy()
        rng.shuffle(class_indices)
        count = len(class_indices)

        if count == 1:
            train_parts.append(class_indices)
            notes.append(f"class {class_id}: only 1 sample -> train only")
            continue

        if count == 2:
            train_parts.append(class_indices[:1])
            val_parts.append(class_indices[1:])
            notes.append(f"class {class_id}: only 2 samples -> train/validation only")
            continue

        train_count = int(round(count * 0.70))
        val_count = int(round(count * 0.15))

        train_count = max(1, train_count)
        val_count = max(1, val_count)

        if train_count + val_count >= count:
            train_count = count - 2
            val_count = 1

        holdout_count = count - train_count - val_count
        if holdout_count < 1:
            holdout_count = 1
            if train_count > val_count:
                train_count -= 1
            else:
                val_count -= 1

        train_parts.append(class_indices[:train_count])
        val_parts.append(class_indices[train_count:train_count + val_count])
        holdout_parts.append(class_indices[train_count + val_count:])

    train_split = np.concatenate(train_parts) if train_parts else np.array([], dtype=np.int64)
    val_split = np.concatenate(val_parts) if val_parts else np.array([], dtype=np.int64)
    holdout_split = np.concatenate(holdout_parts) if holdout_parts else np.array([], dtype=np.int64)

    return train_split, val_split, holdout_split, notes

try:
    train_indices, temp_indices = train_test_split(
        all_indices,
        train_size=0.70,
        random_state=SEED,
        stratify=y_encoded,
    )

    val_indices, holdout_indices = train_test_split(
        temp_indices,
        test_size=0.50,
        random_state=SEED,
        stratify=y_encoded[temp_indices],
    )
except ValueError as error:
    print("Strict sklearn stratified split failed:", error)
    print("Using rare-class-aware stratified fallback.")
    split_method = "rare_class_aware_per_class_stratified_fallback"
    train_indices, val_indices, holdout_indices, rare_class_notes = rare_class_stratified_fallback(
        y_encoded,
        SEED,
    )

train_indices = np.sort(train_indices.astype(np.int64))
val_indices = np.sort(val_indices.astype(np.int64))
holdout_indices = np.sort(holdout_indices.astype(np.int64))

np.save(TRAIN_INDICES_PATH, train_indices)
np.save(VAL_INDICES_PATH, val_indices)
np.save(HOLDOUT_INDICES_PATH, holdout_indices)

def split_row(name, indices):
    labels = y_encoded[indices]
    counts = pd.Series(labels).value_counts()
    return {
        "split": name,
        "num_samples": int(len(indices)),
        "fraction": float(len(indices) / len(y_encoded)),
        "num_classes": int(len(np.unique(labels))),
        "min_segments_per_class": int(counts.min()),
        "max_segments_per_class": int(counts.max()),
        "indices_path": str({
            "train": TRAIN_INDICES_PATH,
            "validation": VAL_INDICES_PATH,
            "holdout": HOLDOUT_INDICES_PATH,
        }[name]),
    }

split_summary = pd.DataFrame([
    split_row("train", train_indices),
    split_row("validation", val_indices),
    split_row("holdout", holdout_indices),
])
split_summary["split_method"] = split_method
split_summary["rare_class_note_count"] = len(rare_class_notes)
split_summary.to_csv(SPLIT_SUMMARY_PATH, index=False)

print(split_summary.to_string(index=False))
if rare_class_notes:
    print("Rare-class handling notes:")
    for note in rare_class_notes[:20]:
        print(" -", note)
    if len(rare_class_notes) > 20:
        print(" - ...", len(rare_class_notes) - 20, "more notes")
print("split summary saved:", SPLIT_SUMMARY_PATH)


Strict sklearn stratified split failed: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [7]
Using rare-class-aware stratified fallback.
     split  num_samples  fraction  num_classes  min_segments_per_class  max_segments_per_class                                                          indices_path                                   split_method  rare_class_note_count
     train        25145  0.696480         1019                       3                     168   F:\ECG\data\processed\feature_outputs_v5\resnet1d_train_indices.npy rare_class_aware_per_class_stratified_fallback                      0
validation         5450  0.150957         1019                       1                      36     F:\ECG\data\processed\feature_outputs_v5\resnet1d_val_indices.npy rare_class_aware_per_class_stratified_fallback                      0
   holdout         5508  0.152563   

## 4. Define the ResNet1D Backbone and Classifier

The model follows the V5 `AGENT.md` design: a 1D convolutional stem, residual blocks, global pooling, an embedding layer, and a classifier head. `forward()` returns 1019-class logits, while `extract_embedding()` returns the embedding vector before the final classifier.


In [15]:
class ECGSegmentDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        row_index = self.indices[item]
        signal = torch.from_numpy(self.X[row_index]).float().unsqueeze(0)
        label = torch.tensor(int(self.y[row_index]), dtype=torch.long)
        return signal, label


class ECGOnlyDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        signal = torch.from_numpy(self.X[item]).float().unsqueeze(0)
        return signal


class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=17, stride=1, dropout=0.3):
        super().__init__()
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)
        out = self.dropout2(out)
        return out


class ResNet1DClassifier(nn.Module):
    def __init__(self, num_classes=1019, kernel_size=17, embedding_dim=128, dropout=0.3):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=kernel_size, stride=1, padding=kernel_size // 2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )

        self.block1 = ResidualBlock1D(16, 16, kernel_size=kernel_size, stride=1, dropout=dropout)
        self.block2 = ResidualBlock1D(16, 32, kernel_size=kernel_size, stride=2, dropout=dropout)
        self.block3 = ResidualBlock1D(32, 64, kernel_size=kernel_size, stride=2, dropout=dropout)
        self.block4 = ResidualBlock1D(64, 128, kernel_size=kernel_size, stride=2, dropout=dropout)

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.embedding_layer = nn.Linear(128, embedding_dim)
        self.embedding_activation = nn.ReLU()
        self.embedding_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def extract_embedding(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)
        embedding = self.embedding_layer(x)
        embedding = self.embedding_activation(embedding)
        return embedding

    def forward(self, x):
        embedding = self.extract_embedding(x)
        logits = self.classifier(self.embedding_dropout(embedding))
        return logits


preview_model = ResNet1DClassifier(
    num_classes=EXPECTED_NUM_CLASSES,
    kernel_size=17,
    embedding_dim=128,
    dropout=0.3,
)
num_parameters = sum(parameter.numel() for parameter in preview_model.parameters())
print(preview_model)
print("number of parameters:", num_parameters)
del preview_model


ResNet1DClassifier(
  (stem): Sequential(
    (0): Conv1d(1, 16, kernel_size=(17,), stride=(1,), padding=(8,))
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block1): ResidualBlock1D(
    (conv1): Conv1d(16, 16, kernel_size=(17,), stride=(1,), padding=(8,))
    (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (dropout1): Dropout(p=0.3, inplace=False)
    (conv2): Conv1d(16, 16, kernel_size=(17,), stride=(1,), padding=(8,))
    (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (dropout2): Dropout(p=0.3, inplace=False)
    (shortcut): Identity()
  )
  (block2): ResidualBlock1D(
    (conv1): Conv1d(16, 32, kernel_size=(17,), stride=(2,), padding=(8,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
 

## 5. Train the UofTDB-specific ResNet1D Subject Classifier

Training task: 1019-class subject classification.

Default settings:

```text
optimizer = Adam
learning_rate = 0.001
batch_size = 64
max_epochs = 80
early_stopping_patience = 10
loss = CrossEntropyLoss
embedding_dim = 128
kernel_size = 17
dropout = 0.3
```

If CUDA runs out of memory, the notebook switches to `batch_size = 32` and `max_epochs = 50`. The actual settings are written to `resnet1d_config.json`.


In [18]:
BASE_CONFIG = {
    "seed": SEED,
    "task": "1019-class UofTDB subject classification",
    "input_shape": "[batch, 1, 1000]",
    "kernel_size": 17,
    "embedding_dim": 128,
    "dropout": 0.3,
    "activation": "ReLU",
    "num_classes": EXPECTED_NUM_CLASSES,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "CrossEntropyLoss",
    "requested_batch_size": 64,
    "requested_max_epochs": 80,
    "early_stopping_patience": 10,
    "scheduler": "ReduceLROnPlateau(factor=0.5, patience=4)",
    "oom_fallback_batch_size": 32,
    "oom_fallback_max_epochs": 50,
    "device": str(device),
    "gpu_name": gpu_name,
    "torch_version": torch.__version__,
    "training_note": "This split trains only the offline ResNet1D embedding extractor; RanPAC later uses the full time-aware stream.",
}

def make_loaders(batch_size):
    pin_memory = device.type == "cuda"
    train_dataset = ECGSegmentDataset(X_stream, y_encoded, train_indices)
    val_dataset = ECGSegmentDataset(X_stream, y_encoded, val_indices)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=pin_memory,
    )
    return train_loader, val_loader


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for signals, labels in loader:
        signals = signals.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(signals)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        predictions = torch.argmax(logits, dim=1)
        correct += int((predictions == labels).sum().item())
        total += int(batch_size)

    return total_loss / total, correct / total


def evaluate_loss_accuracy(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for signals, labels in loader:
            signals = signals.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(signals)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += float(loss.item()) * batch_size
            predictions = torch.argmax(logits, dim=1)
            correct += int((predictions == labels).sum().item())
            total += int(batch_size)

    return total_loss / total, correct / total


def run_training(batch_size, max_epochs):
    set_all_seeds(SEED)

    model = ResNet1DClassifier(
        num_classes=EXPECTED_NUM_CLASSES,
        kernel_size=BASE_CONFIG["kernel_size"],
        embedding_dim=BASE_CONFIG["embedding_dim"],
        dropout=BASE_CONFIG["dropout"],
    ).to(device)

    train_loader, val_loader = make_loaders(batch_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=BASE_CONFIG["learning_rate"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=4,
    )

    best_val_loss = float("inf")
    best_epoch = 0
    best_val_accuracy = 0.0
    epochs_without_improvement = 0
    training_rows = []
    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = evaluate_loss_accuracy(model, val_loader, criterion)

        current_lr = float(optimizer.param_groups[0]["lr"])
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_accuracy,
            "learning_rate": current_lr,
        }
        training_rows.append(row)

        print(
            f"epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} val_acc={val_accuracy:.4f} | "
            f"lr={current_lr:.6f}"
        )

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_val_accuracy = val_accuracy
            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "model_state_dict": model.state_dict(),
                "num_classes": EXPECTED_NUM_CLASSES,
                "kernel_size": BASE_CONFIG["kernel_size"],
                "embedding_dim": BASE_CONFIG["embedding_dim"],
                "dropout": BASE_CONFIG["dropout"],
                "best_epoch": best_epoch,
                "best_val_loss": best_val_loss,
                "best_val_accuracy": best_val_accuracy,
                "subject_label_mapping_path": str(LABEL_MAPPING_PATH),
            }
            torch.save(checkpoint, BEST_MODEL_PATH)
        else:
            epochs_without_improvement += 1

        scheduler.step(val_loss)

        if epochs_without_improvement >= BASE_CONFIG["early_stopping_patience"]:
            print("Early stopping triggered at epoch", epoch)
            break

    elapsed_seconds = time.time() - start_time
    training_log = pd.DataFrame(training_rows)
    best_result = {
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "best_val_accuracy": float(best_val_accuracy),
        "epochs_completed": int(len(training_rows)),
        "training_seconds": float(elapsed_seconds),
    }

    return model, training_log, best_result


actual_batch_size = BASE_CONFIG["requested_batch_size"]
actual_max_epochs = BASE_CONFIG["requested_max_epochs"]
used_oom_fallback = False

try:
    model, training_log, best_result = run_training(
        batch_size=actual_batch_size,
        max_epochs=actual_max_epochs,
    )
except RuntimeError as error:
    error_text = str(error).lower()
    if device.type == "cuda" and "out of memory" in error_text:
        print("CUDA out-of-memory detected. Retrying with smaller settings.")
        torch.cuda.empty_cache()
        actual_batch_size = BASE_CONFIG["oom_fallback_batch_size"]
        actual_max_epochs = BASE_CONFIG["oom_fallback_max_epochs"]
        used_oom_fallback = True
        model, training_log, best_result = run_training(
            batch_size=actual_batch_size,
            max_epochs=actual_max_epochs,
        )
    else:
        raise

training_log.to_csv(TRAINING_LOG_PATH, index=False)

actual_config = dict(BASE_CONFIG)
actual_config.update({
    "actual_batch_size": int(actual_batch_size),
    "actual_max_epochs": int(actual_max_epochs),
    "used_oom_fallback": bool(used_oom_fallback),
    "best_model_path": str(BEST_MODEL_PATH),
    "training_log_path": str(TRAINING_LOG_PATH),
    "best_epoch": int(best_result["best_epoch"]),
    "best_val_loss": float(best_result["best_val_loss"]),
    "best_val_accuracy": float(best_result["best_val_accuracy"]),
    "epochs_completed": int(best_result["epochs_completed"]),
    "training_seconds": float(best_result["training_seconds"]),
})

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(actual_config, file, indent=2, ensure_ascii=False)

print("best model saved:", BEST_MODEL_PATH)
print("training log saved:", TRAINING_LOG_PATH)
print("config saved:", CONFIG_PATH)
print("best result:", best_result)


epoch 001 | train_loss=5.7867 val_loss=5.0156 | train_acc=0.0286 val_acc=0.0534 | lr=0.001000
epoch 002 | train_loss=4.5397 val_loss=3.8090 | train_acc=0.0815 val_acc=0.1609 | lr=0.001000
epoch 003 | train_loss=3.6871 val_loss=2.8571 | train_acc=0.1629 val_acc=0.2985 | lr=0.001000
epoch 004 | train_loss=3.0063 val_loss=2.2992 | train_acc=0.2612 val_acc=0.4011 | lr=0.001000
epoch 005 | train_loss=2.5052 val_loss=1.7484 | train_acc=0.3442 val_acc=0.5097 | lr=0.001000
epoch 006 | train_loss=2.1495 val_loss=1.5413 | train_acc=0.4130 val_acc=0.5637 | lr=0.001000
epoch 007 | train_loss=1.8888 val_loss=1.4524 | train_acc=0.4651 val_acc=0.5921 | lr=0.001000
epoch 008 | train_loss=1.7026 val_loss=1.1621 | train_acc=0.5106 val_acc=0.6747 | lr=0.001000
epoch 009 | train_loss=1.5392 val_loss=0.9464 | train_acc=0.5516 val_acc=0.7218 | lr=0.001000
epoch 010 | train_loss=1.4243 val_loss=0.8319 | train_acc=0.5810 val_acc=0.7593 | lr=0.001000
epoch 011 | train_loss=1.2991 val_loss=0.6914 | train_acc=0.

## 6. Evaluate the Holdout Set for Backbone Sanity Check

The holdout set is used only to check whether the ResNet1D backbone learned subject-discriminative representations. It is not the main thesis result and not the RanPAC online evaluation split.

If holdout accuracy is close to the random baseline `1/1019`, or if validation loss does not decrease, the notebook prints a clear warning.


In [21]:
def load_best_model():
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    loaded_model = ResNet1DClassifier(
        num_classes=checkpoint["num_classes"],
        kernel_size=checkpoint["kernel_size"],
        embedding_dim=checkpoint["embedding_dim"],
        dropout=checkpoint["dropout"],
    ).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    return loaded_model, checkpoint


def predict_for_indices(model, indices, batch_size):
    dataset = ECGSegmentDataset(X_stream, y_encoded, indices)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for signals, labels in loader:
            signals = signals.to(device, non_blocking=True)
            logits = model(signals)
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            all_predictions.append(predictions)
            all_labels.append(labels.numpy())

    return np.concatenate(all_labels), np.concatenate(all_predictions)


best_model, best_checkpoint = load_best_model()

holdout_labels, holdout_predictions = predict_for_indices(
    best_model,
    holdout_indices,
    batch_size=actual_batch_size,
)

holdout_accuracy = accuracy_score(holdout_labels, holdout_predictions)
macro_precision = precision_score(
    holdout_labels,
    holdout_predictions,
    labels=np.arange(EXPECTED_NUM_CLASSES),
    average="macro",
    zero_division=0,
)
macro_f1 = f1_score(
    holdout_labels,
    holdout_predictions,
    labels=np.arange(EXPECTED_NUM_CLASSES),
    average="macro",
    zero_division=0,
)

random_baseline = 1.0 / EXPECTED_NUM_CLASSES
validation_loss_drop = float(training_log["val_loss"].iloc[0] - training_log["val_loss"].min())

warnings_list = []
if holdout_accuracy <= random_baseline * 3:
    warnings_list.append(
        "WARNING: holdout accuracy is close to random baseline 1/1019. Do not treat this embedding as qualified without further inspection."
    )
if validation_loss_drop <= 0:
    warnings_list.append(
        "WARNING: validation loss did not decrease. Do not treat this embedding as qualified without further inspection."
    )

holdout_report = pd.DataFrame([{
    "holdout_accuracy": float(holdout_accuracy),
    "macro_precision": float(macro_precision),
    "macro_f1": float(macro_f1),
    "num_samples": int(len(holdout_indices)),
    "num_classes": int(EXPECTED_NUM_CLASSES),
    "random_baseline_accuracy": float(random_baseline),
    "validation_loss_drop": float(validation_loss_drop),
    "warnings": " | ".join(warnings_list) if warnings_list else "",
}])
holdout_report.to_csv(HOLDOUT_REPORT_PATH, index=False)

print(holdout_report.to_string(index=False))
for warning_text in warnings_list:
    print(warning_text)
print("holdout report saved:", HOLDOUT_REPORT_PATH)


 holdout_accuracy  macro_precision  macro_f1  num_samples  num_classes  random_baseline_accuracy  validation_loss_drop warnings
         0.959513         0.959225  0.945975         5508         1019                  0.000981              4.863539         
holdout report saved: F:\ECG\data\processed\feature_outputs_v5\resnet1d_holdout_report.csv


## 7. Load the Best Model, Freeze ResNet1D, and Extract Full-stream Embeddings

This section reloads `resnet1d_uoftdb_best.pth`, freezes all parameters, and uses only `extract_embedding()`. The full `X_stream` is passed through the backbone to generate `X_features_embedding_resnet1d.npy`.


In [24]:
embedding_model, embedding_checkpoint = load_best_model()
embedding_model.eval()
for parameter in embedding_model.parameters():
    parameter.requires_grad = False

embedding_batch_size = 256 if device.type == "cuda" else 64
embedding_dataset = ECGOnlyDataset(X_stream)
embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=embedding_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

embedding_batches = []
with torch.no_grad():
    for signals in embedding_loader:
        signals = signals.to(device, non_blocking=True)
        batch_embeddings = embedding_model.extract_embedding(signals)
        embedding_batches.append(batch_embeddings.cpu().numpy())

X_features_embedding_resnet1d = np.vstack(embedding_batches).astype(np.float32)
np.save(EMBEDDING_FEATURE_PATH, X_features_embedding_resnet1d)

embedding_report = pd.DataFrame([{
    "num_samples": int(X_features_embedding_resnet1d.shape[0]),
    "embedding_dim": int(X_features_embedding_resnet1d.shape[1]),
    "checkpoint_path": str(BEST_MODEL_PATH),
    "contains_nan": bool(np.isnan(X_features_embedding_resnet1d).any()),
    "contains_inf": bool(np.isinf(X_features_embedding_resnet1d).any()),
    "training_source": "UofTDB self-trained subject classification",
    "input_length": int(EXPECTED_SIGNAL_LENGTH),
    "extraction_device": str(device),
    "embedding_batch_size": int(embedding_batch_size),
}])
embedding_report.to_csv(EMBEDDING_REPORT_PATH, index=False)

print("embedding shape:", X_features_embedding_resnet1d.shape)
print(embedding_report.to_string(index=False))
print("embedding features saved:", EMBEDDING_FEATURE_PATH)
print("embedding report saved:", EMBEDDING_REPORT_PATH)

if np.isnan(X_features_embedding_resnet1d).any() or np.isinf(X_features_embedding_resnet1d).any():
    raise ValueError("Embedding feature matrix contains NaN or Inf.")


embedding shape: (36103, 128)
 num_samples  embedding_dim                                                   checkpoint_path  contains_nan  contains_inf                            training_source  input_length extraction_device  embedding_batch_size
       36103            128 F:\ECG\data\processed\feature_outputs_v5\resnet1d_uoftdb_best.pth         False         False UofTDB self-trained subject classification          1000              cuda                   256
embedding features saved: F:\ECG\data\processed\feature_outputs_v5\X_features_embedding_resnet1d.npy
embedding report saved: F:\ECG\data\processed\feature_outputs_v5\embedding_resnet1d_report.csv


## 8. Extract PQRST-oriented Fiducial Features from the Same X_stream

NeuroKit2 is used when available:

```text
ecg_clean
ecg_peaks
ecg_delineate
sampling_rate = 200
```

If NeuroKit2 is unavailable or fails for a segment, a scipy fallback extracts R-peak / QRS-oriented features. The fallback is not full PQRST, so the notebook and quality report do not claim full PQRST success without evidence.

The final feature matrix must not contain NaN or Inf. Missing P/Q/S/T or interval features use controlled column-median imputation; if an entire column is missing, it is filled with 0 and recorded in the quality report.


In [27]:
FIDUCIAL_FEATURE_NAMES = [
    "r_peak_count",
    "valid_beat_count",
    "rr_mean",
    "rr_std",
    "rr_min",
    "rr_max",
    "heart_rate_bpm",
    "p_amp_mean",
    "q_amp_mean",
    "r_amp_mean",
    "s_amp_mean",
    "t_amp_mean",
    "qrs_duration_mean",
    "qrs_duration_std",
    "pr_interval_mean",
    "qt_interval_mean",
    "segment_mean",
    "segment_std",
    "segment_min",
    "segment_max",
    "rms",
    "energy",
    "pqrst_detection_success_rate",
]

def empty_feature_dict(segment):
    features = {name: np.nan for name in FIDUCIAL_FEATURE_NAMES}
    segment = np.asarray(segment, dtype=float)
    features["segment_mean"] = float(np.mean(segment))
    features["segment_std"] = float(np.std(segment))
    features["segment_min"] = float(np.min(segment))
    features["segment_max"] = float(np.max(segment))
    features["rms"] = float(np.sqrt(np.mean(segment ** 2)))
    features["energy"] = float(np.sum(segment ** 2))
    features["r_peak_count"] = 0.0
    features["valid_beat_count"] = 0.0
    features["pqrst_detection_success_rate"] = 0.0
    return features


def valid_indices(values, signal_length):
    if values is None:
        return np.array([], dtype=np.int64)
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[(values >= 0) & (values < signal_length)]
    return values.astype(np.int64)


def mean_at_indices(signal, indices):
    if len(indices) == 0:
        return np.nan
    return float(np.mean(signal[indices]))


def positive_intervals(left_indices, right_indices, sampling_rate):
    left_indices = np.asarray(left_indices, dtype=float)
    right_indices = np.asarray(right_indices, dtype=float)
    pair_count = min(len(left_indices), len(right_indices))
    if pair_count == 0:
        return np.array([], dtype=float)
    values = (right_indices[:pair_count] - left_indices[:pair_count]) / sampling_rate
    values = values[np.isfinite(values)]
    values = values[(values > 0) & (values < 2.5)]
    return values


def fill_rr_features(features, rpeaks, signal):
    rpeaks = np.asarray(rpeaks, dtype=np.int64)
    features["r_peak_count"] = float(len(rpeaks))
    features["r_amp_mean"] = mean_at_indices(signal, rpeaks)

    if len(rpeaks) >= 2:
        rr_intervals = np.diff(rpeaks) / SAMPLING_RATE
        rr_intervals = rr_intervals[np.isfinite(rr_intervals)]
        rr_intervals = rr_intervals[rr_intervals > 0]
        if len(rr_intervals) > 0:
            features["rr_mean"] = float(np.mean(rr_intervals))
            features["rr_std"] = float(np.std(rr_intervals))
            features["rr_min"] = float(np.min(rr_intervals))
            features["rr_max"] = float(np.max(rr_intervals))
            features["heart_rate_bpm"] = float(60.0 / np.mean(rr_intervals))


def scipy_fallback_features(segment):
    features = empty_feature_dict(segment)
    quality = {
        "used_fallback": True,
        "fallback_reason": "neurokit_unavailable_or_failed",
        "r_peak_count": 0,
        "valid_beat_count": 0,
        "pqrst_detection_success_rate": 0.0,
    }

    if not SCIPY_AVAILABLE:
        quality["fallback_reason"] = "scipy_unavailable"
        return features, quality

    segment = np.asarray(segment, dtype=float)
    signal_std = float(np.std(segment))
    prominence = max(0.20 * signal_std, 0.05)
    min_distance = int(0.25 * SAMPLING_RATE)
    peaks, _ = find_peaks(segment, distance=min_distance, prominence=prominence)
    rpeaks = valid_indices(peaks, len(segment))
    fill_rr_features(features, rpeaks, segment)

    q_indices = []
    s_indices = []
    search_width = int(0.08 * SAMPLING_RATE)
    for r_index in rpeaks:
        q_start = max(0, r_index - search_width)
        q_end = r_index
        s_start = r_index
        s_end = min(len(segment), r_index + search_width + 1)

        if q_end > q_start:
            q_indices.append(q_start + int(np.argmin(segment[q_start:q_end])))
        if s_end > s_start:
            s_indices.append(s_start + int(np.argmin(segment[s_start:s_end])))

    q_indices = np.asarray(q_indices, dtype=np.int64)
    s_indices = np.asarray(s_indices, dtype=np.int64)
    features["q_amp_mean"] = mean_at_indices(segment, q_indices)
    features["s_amp_mean"] = mean_at_indices(segment, s_indices)

    qrs_intervals = positive_intervals(q_indices, s_indices, SAMPLING_RATE)
    if len(qrs_intervals) > 0:
        features["qrs_duration_mean"] = float(np.mean(qrs_intervals))
        features["qrs_duration_std"] = float(np.std(qrs_intervals))

    features["valid_beat_count"] = float(min(len(q_indices), len(rpeaks), len(s_indices)))
    features["pqrst_detection_success_rate"] = 0.0

    quality["r_peak_count"] = int(features["r_peak_count"])
    quality["valid_beat_count"] = int(features["valid_beat_count"])
    quality["pqrst_detection_success_rate"] = float(features["pqrst_detection_success_rate"])
    return features, quality


def extract_fiducial_one(row_index):
    segment = np.asarray(X_stream[row_index], dtype=float)
    features = empty_feature_dict(segment)
    quality = {
        "row_index": int(row_index),
        "used_neurokit": False,
        "neurokit_success": False,
        "delineation_success": False,
        "used_fallback": False,
        "error_message": "",
        "r_peak_count": 0,
        "valid_beat_count": 0,
        "pqrst_detection_success_rate": 0.0,
    }

    if NK_AVAILABLE:
        try:
            cleaned = nk.ecg_clean(segment, sampling_rate=SAMPLING_RATE, method="neurokit")
            _, peak_info = nk.ecg_peaks(cleaned, sampling_rate=SAMPLING_RATE)
            rpeaks = valid_indices(peak_info.get("ECG_R_Peaks", []), len(cleaned))
            fill_rr_features(features, rpeaks, cleaned)
            quality["used_neurokit"] = True
            quality["neurokit_success"] = True

            if len(rpeaks) > 0:
                _, waves = nk.ecg_delineate(
                    cleaned,
                    rpeaks=rpeaks,
                    sampling_rate=SAMPLING_RATE,
                    method="dwt",
                )
                if isinstance(waves, dict):
                    p_peaks = valid_indices(waves.get("ECG_P_Peaks"), len(cleaned))
                    q_peaks = valid_indices(waves.get("ECG_Q_Peaks"), len(cleaned))
                    s_peaks = valid_indices(waves.get("ECG_S_Peaks"), len(cleaned))
                    t_peaks = valid_indices(waves.get("ECG_T_Peaks"), len(cleaned))

                    p_onsets = valid_indices(waves.get("ECG_P_Onsets"), len(cleaned))
                    r_onsets = valid_indices(waves.get("ECG_R_Onsets"), len(cleaned))
                    r_offsets = valid_indices(waves.get("ECG_R_Offsets"), len(cleaned))
                    t_offsets = valid_indices(waves.get("ECG_T_Offsets"), len(cleaned))

                    features["p_amp_mean"] = mean_at_indices(cleaned, p_peaks)
                    features["q_amp_mean"] = mean_at_indices(cleaned, q_peaks)
                    features["s_amp_mean"] = mean_at_indices(cleaned, s_peaks)
                    features["t_amp_mean"] = mean_at_indices(cleaned, t_peaks)

                    qrs_intervals = positive_intervals(r_onsets, r_offsets, SAMPLING_RATE)
                    if len(qrs_intervals) == 0:
                        qrs_intervals = positive_intervals(q_peaks, s_peaks, SAMPLING_RATE)
                    if len(qrs_intervals) > 0:
                        features["qrs_duration_mean"] = float(np.mean(qrs_intervals))
                        features["qrs_duration_std"] = float(np.std(qrs_intervals))

                    pr_intervals = positive_intervals(p_onsets, r_onsets, SAMPLING_RATE)
                    if len(pr_intervals) > 0:
                        features["pr_interval_mean"] = float(np.mean(pr_intervals))

                    qt_intervals = positive_intervals(r_onsets, t_offsets, SAMPLING_RATE)
                    if len(qt_intervals) > 0:
                        features["qt_interval_mean"] = float(np.mean(qt_intervals))

                    valid_beat_count = min(
                        len(p_peaks),
                        len(q_peaks),
                        len(rpeaks),
                        len(s_peaks),
                        len(t_peaks),
                    )
                    features["valid_beat_count"] = float(valid_beat_count)
                    features["pqrst_detection_success_rate"] = float(valid_beat_count / len(rpeaks))
                    quality["delineation_success"] = True

            quality["r_peak_count"] = int(features["r_peak_count"])
            quality["valid_beat_count"] = int(features["valid_beat_count"])
            quality["pqrst_detection_success_rate"] = float(features["pqrst_detection_success_rate"])

            if int(features["r_peak_count"]) > 0:
                return [features[name] for name in FIDUCIAL_FEATURE_NAMES], quality

            quality["error_message"] = "NeuroKit2 returned zero R peaks; scipy fallback used."

        except Exception as error:
            quality["error_message"] = repr(error)[:300]

    fallback_features, fallback_quality = scipy_fallback_features(segment)
    quality.update(fallback_quality)
    quality["used_fallback"] = True
    if quality["error_message"] == "":
        quality["error_message"] = fallback_quality.get("fallback_reason", "")

    return [fallback_features[name] for name in FIDUCIAL_FEATURE_NAMES], quality


fiducial_start_time = time.time()
fiducial_n_jobs = min(4, os.cpu_count() or 1)
print("Starting fiducial extraction with n_jobs =", fiducial_n_jobs)
print("NeuroKit2 available:", NK_AVAILABLE, "version:", NEUROKIT_VERSION)

fiducial_pairs = Parallel(n_jobs=fiducial_n_jobs, prefer="threads", verbose=5)(
    delayed(extract_fiducial_one)(row_index)
    for row_index in range(len(X_stream))
)

fiducial_elapsed_seconds = time.time() - fiducial_start_time

fiducial_matrix_raw = np.asarray([pair[0] for pair in fiducial_pairs], dtype=np.float64)
fiducial_quality_detail = pd.DataFrame([pair[1] for pair in fiducial_pairs])

fiducial_df = pd.DataFrame(fiducial_matrix_raw, columns=FIDUCIAL_FEATURE_NAMES)
fiducial_df = fiducial_df.replace([np.inf, -np.inf], np.nan)
missing_before_imputation = fiducial_df.isna().sum()

imputation_values = {}
for feature_name in FIDUCIAL_FEATURE_NAMES:
    column_median = fiducial_df[feature_name].median(skipna=True)
    if pd.isna(column_median):
        column_median = 0.0
    imputation_values[feature_name] = float(column_median)
    fiducial_df[feature_name] = fiducial_df[feature_name].fillna(column_median)

X_features_fiducial_pqrst = fiducial_df.to_numpy(dtype=np.float32)

if np.isnan(X_features_fiducial_pqrst).any() or np.isinf(X_features_fiducial_pqrst).any():
    raise ValueError("Fiducial feature matrix still contains NaN or Inf after controlled imputation.")

np.save(FIDUCIAL_FEATURE_PATH, X_features_fiducial_pqrst)
pd.DataFrame({
    "feature_index": np.arange(len(FIDUCIAL_FEATURE_NAMES), dtype=int),
    "feature_name": FIDUCIAL_FEATURE_NAMES,
}).to_csv(FIDUCIAL_NAMES_PATH, index=False)

quality_rows = [
    {"section": "pipeline", "metric": "neurokit_available", "value": bool(NK_AVAILABLE), "notes": str(NEUROKIT_VERSION)},
    {"section": "pipeline", "metric": "scipy_fallback_available", "value": bool(SCIPY_AVAILABLE), "notes": ""},
    {"section": "pipeline", "metric": "fiducial_n_jobs", "value": int(fiducial_n_jobs), "notes": ""},
    {"section": "pipeline", "metric": "elapsed_seconds", "value": float(fiducial_elapsed_seconds), "notes": ""},
    {"section": "detection", "metric": "num_segments", "value": int(len(X_stream)), "notes": ""},
    {"section": "detection", "metric": "neurokit_success_segments", "value": int(fiducial_quality_detail["neurokit_success"].sum()), "notes": ""},
    {"section": "detection", "metric": "delineation_success_segments", "value": int(fiducial_quality_detail["delineation_success"].sum()), "notes": ""},
    {"section": "detection", "metric": "fallback_segments", "value": int(fiducial_quality_detail["used_fallback"].sum()), "notes": "Fallback is R-peak/QRS-oriented, not full PQRST."},
    {"section": "detection", "metric": "mean_r_peak_count", "value": float(fiducial_quality_detail["r_peak_count"].mean()), "notes": ""},
    {"section": "detection", "metric": "mean_valid_beat_count", "value": float(fiducial_quality_detail["valid_beat_count"].mean()), "notes": ""},
    {"section": "detection", "metric": "mean_pqrst_detection_success_rate", "value": float(fiducial_quality_detail["pqrst_detection_success_rate"].mean()), "notes": "Average of per-segment valid PQRST beat count divided by R-peak count."},
    {"section": "detection", "metric": "segments_with_any_valid_pqrst", "value": int((fiducial_quality_detail["valid_beat_count"] > 0).sum()), "notes": ""},
]

for feature_name in FIDUCIAL_FEATURE_NAMES:
    quality_rows.append({
        "section": "missingness_before_imputation",
        "metric": feature_name,
        "value": int(missing_before_imputation[feature_name]),
        "notes": f"missing_fraction={missing_before_imputation[feature_name] / len(fiducial_df):.6f}; imputation_value={imputation_values[feature_name]:.6f}",
    })

fiducial_quality_report = pd.DataFrame(quality_rows)
fiducial_quality_report.to_csv(FIDUCIAL_QUALITY_PATH, index=False)

print("fiducial shape:", X_features_fiducial_pqrst.shape)
print("fiducial extraction seconds:", round(fiducial_elapsed_seconds, 2))
print(fiducial_quality_report.to_string(index=False))
print("fiducial features saved:", FIDUCIAL_FEATURE_PATH)
print("fiducial feature names saved:", FIDUCIAL_NAMES_PATH)
print("fiducial quality report saved:", FIDUCIAL_QUALITY_PATH)


Starting fiducial extraction with n_jobs = 4
NeuroKit2 available: True version: 0.2.13


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    4.0s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done 280 tasks      | elapsed:   13.0s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:   20.0s
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed:   30.2s
[Parallel(n_jobs=4)]: Done 874 tasks      | elapsed:   41.0s
[Parallel(n_jobs=4)]: Done 1144 tasks      | elapsed:   53.5s
[Parallel(n_jobs=4)]: Done 1450 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done 2170 tasks      | elapsed:  1.7min
[Parallel(n_jobs=4)]: Done 2584 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done 3034 tasks      | elapsed:  2.3min
[Parallel(n_jobs=4)]: Done 3520 tasks      | elapsed:  2.8min
[Parallel(n_jobs=4)]: Done 4042 tasks      | elapsed:  3.3m

fiducial shape: (36103, 23)
fiducial extraction seconds: 1566.69
                      section                            metric        value                                                                  notes
                     pipeline                neurokit_available         True                                                                 0.2.13
                     pipeline          scipy_fallback_available         True                                                                       
                     pipeline                   fiducial_n_jobs            4                                                                       
                     pipeline                   elapsed_seconds  1566.690923                                                                       
                    detection                      num_segments        36103                                                                       
                    detection         neurokit_

[Parallel(n_jobs=4)]: Done 36103 out of 36103 | elapsed: 26.1min finished


## 9. Standardize Fiducial and Embedding Features, then Concatenate Both

Fiducial and embedding features are standardized separately before concatenation:

```text
X_both = concat(standardized_fiducial, standardized_embedding)
```

The notebook saves the three feature matrices, the feature summary, and the encoded `y_stream` for Step 6 and Step 7. It still does not run Random Projection or RanPAC.


In [30]:
fiducial_scaler = StandardScaler()
embedding_scaler = StandardScaler()

standardized_fiducial = fiducial_scaler.fit_transform(X_features_fiducial_pqrst).astype(np.float32)
standardized_embedding = embedding_scaler.fit_transform(X_features_embedding_resnet1d).astype(np.float32)

X_features_both_pqrst_resnet1d = np.concatenate(
    [standardized_fiducial, standardized_embedding],
    axis=1,
).astype(np.float32)

np.save(BOTH_FEATURE_PATH, X_features_both_pqrst_resnet1d)
np.save(Y_STREAM_OUTPUT_PATH, y_encoded.astype(np.int64))

def feature_summary_row(feature_setting, matrix, path, notes):
    return {
        "feature_setting": feature_setting,
        "num_samples": int(matrix.shape[0]),
        "feature_dim": int(matrix.shape[1]),
        "contains_nan": bool(np.isnan(matrix).any()),
        "contains_inf": bool(np.isinf(matrix).any()),
        "output_path": str(path),
        "notes": notes,
    }

feature_summary = pd.DataFrame([
    feature_summary_row(
        "fiducial_pqrst",
        X_features_fiducial_pqrst,
        FIDUCIAL_FEATURE_PATH,
        "PQRST-oriented fiducial features with controlled median/zero imputation for missing values.",
    ),
    feature_summary_row(
        "embedding_resnet1d",
        X_features_embedding_resnet1d,
        EMBEDDING_FEATURE_PATH,
        "Embedding from UofTDB self-trained ResNet1D subject classifier; classifier head is not used.",
    ),
    feature_summary_row(
        "both_pqrst_resnet1d",
        X_features_both_pqrst_resnet1d,
        BOTH_FEATURE_PATH,
        "Concatenation of separately standardized fiducial and ResNet1D embedding features.",
    ),
])
feature_summary.to_csv(FEATURE_SUMMARY_PATH, index=False)

print(feature_summary.to_string(index=False))
print("both features saved:", BOTH_FEATURE_PATH)
print("encoded y_stream saved:", Y_STREAM_OUTPUT_PATH)
print("feature summary saved:", FEATURE_SUMMARY_PATH)

if not bool((feature_summary["num_samples"] == EXPECTED_NUM_SAMPLES).all()):
    raise ValueError("At least one feature matrix does not have 36103 rows.")
if bool(feature_summary["contains_nan"].any()) or bool(feature_summary["contains_inf"].any()):
    raise ValueError("At least one feature matrix contains NaN or Inf.")


    feature_setting  num_samples  feature_dim  contains_nan  contains_inf                                                                 output_path                                                                                        notes
     fiducial_pqrst        36103           23         False         False      F:\ECG\data\processed\feature_outputs_v5\X_features_fiducial_pqrst.npy  PQRST-oriented fiducial features with controlled median/zero imputation for missing values.
 embedding_resnet1d        36103          128         False         False  F:\ECG\data\processed\feature_outputs_v5\X_features_embedding_resnet1d.npy Embedding from UofTDB self-trained ResNet1D subject classifier; classifier head is not used.
both_pqrst_resnet1d        36103          151         False         False F:\ECG\data\processed\feature_outputs_v5\X_features_both_pqrst_resnet1d.npy           Concatenation of separately standardized fiducial and ResNet1D embedding features.
both features saved: F:\ECG\

## 10. Final Step 5 Validation and Output Summary

The final section prints key Step 5 results and all output paths. If any row-count or NaN/Inf validation fails, do not proceed to Step 6.


In [33]:
best_log_row = training_log.loc[training_log["val_loss"].idxmin()]

final_rows = [
    {"item": "X_stream shape", "value": str(X_stream.shape)},
    {"item": "y_stream shape", "value": str(y_stream.shape)},
    {"item": "unique subject count", "value": int(len(unique_subjects))},
    {"item": "ResNet1D train split", "value": int(len(train_indices))},
    {"item": "ResNet1D validation split", "value": int(len(val_indices))},
    {"item": "ResNet1D holdout split", "value": int(len(holdout_indices))},
    {"item": "best validation epoch", "value": int(best_log_row["epoch"])},
    {"item": "best validation loss", "value": float(best_log_row["val_loss"])},
    {"item": "best validation accuracy", "value": float(best_log_row["val_accuracy"])},
    {"item": "holdout accuracy", "value": float(holdout_accuracy)},
    {"item": "holdout macro-F1", "value": float(macro_f1)},
    {"item": "X_features_fiducial_pqrst shape", "value": str(X_features_fiducial_pqrst.shape)},
    {"item": "X_features_embedding_resnet1d shape", "value": str(X_features_embedding_resnet1d.shape)},
    {"item": "X_features_both_pqrst_resnet1d shape", "value": str(X_features_both_pqrst_resnet1d.shape)},
    {"item": "all three feature matrices have 36103 rows", "value": bool((feature_summary["num_samples"] == EXPECTED_NUM_SAMPLES).all())},
    {"item": "all three feature matrices have no NaN", "value": bool(not feature_summary["contains_nan"].any())},
    {"item": "all three feature matrices have no Inf", "value": bool(not feature_summary["contains_inf"].any())},
]

final_summary = pd.DataFrame(final_rows)
print(final_summary.to_string(index=False))

output_paths = [
    LABEL_MAPPING_PATH,
    TRAIN_INDICES_PATH,
    VAL_INDICES_PATH,
    HOLDOUT_INDICES_PATH,
    SPLIT_SUMMARY_PATH,
    BEST_MODEL_PATH,
    TRAINING_LOG_PATH,
    CONFIG_PATH,
    HOLDOUT_REPORT_PATH,
    EMBEDDING_FEATURE_PATH,
    EMBEDDING_REPORT_PATH,
    FIDUCIAL_FEATURE_PATH,
    FIDUCIAL_NAMES_PATH,
    FIDUCIAL_QUALITY_PATH,
    BOTH_FEATURE_PATH,
    Y_STREAM_OUTPUT_PATH,
    FEATURE_SUMMARY_PATH,
]

print("\nAll Step 5 V5 output files:")
for path in output_paths:
    print(path, "exists =", path.exists())

if not all(path.exists() for path in output_paths):
    raise FileNotFoundError("Some Step 5 output files are missing.")


                                      item         value
                            X_stream shape (36103, 1000)
                            y_stream shape      (36103,)
                      unique subject count          1019
                      ResNet1D train split         25145
                 ResNet1D validation split          5450
                    ResNet1D holdout split          5508
                     best validation epoch            78
                      best validation loss      0.152026
                  best validation accuracy      0.959817
                          holdout accuracy      0.959513
                          holdout macro-F1      0.945975
           X_features_fiducial_pqrst shape   (36103, 23)
       X_features_embedding_resnet1d shape  (36103, 128)
      X_features_both_pqrst_resnet1d shape  (36103, 151)
all three feature matrices have 36103 rows          True
    all three feature matrices have no NaN          True
    all three feature matrices 